In [1]:
%run Latex_macros.ipynb

<IPython.core.display.Latex object>

**Reference**

[SWE-agent: Agent-Computer Interfaces Enable Automated Software Engineering](https://arxiv.org/abs/2405.15793)
-  published by the Princeton NLP Group in 2024.



# The Harness

The traditional interaction between Human and LLM is
- a multi-turn chat

Human asks, LLM reacts, cycle repeats
- Human in the Loop

Role | Content
:--- | :--- |
User | Run the code
Assistant | Result: `AuthError` 
User | Fix the error
Assistant | Fixed, I ran it again, here is the result.
User | But there is now a new error. Fix it
Assisant | ..

We want this to evolve into an interaction that 
- removes the Human from the Loop
- by creating a system with *Agency*
    - something that replaces the Human
    - with a Software Environment
    - that acts on behalf of (as an Agent) of the Human
        - with similar ability to *interact with the Environment (web browser, read/write files)
        - with similar permissions

The Harness is the **Software Environment** 
- that wraps an LLM 
- to transform it from a "Chatbot" into an "Agent."

Integrating a Harness and an LLM is *the beginning* of creating an Agent.


The Harness replaces the Human as the partner is the loop with the LLM.

The ultimate Goal is to create a true Agent:
- Human initiates conversation with Harness
- Harness engages in *multi-turn* conversation with LLM
- Harness responds to Human with result at conclusion of conversation with LLM

Role | Content
:--- | :--- |
User | Run the code
Harness | Run the code with the following goals and constraints
Assistant | Result: `AuthError`
Harness | Find where this exception could have occurred
Assistant | Please run the following `grep` command
          | in order for me to find where `AuthError` occurs in the code base
Harness   | Here is the result of `grep`
Assistant | Exception thrown on line 23 of file `middleware.py`
....
Harness -> Human | Done

## Harness: Generation 1

The first generation of Harness facilitated *limited* Agency
- LLM was able to interact with Environment
- with Human still in the Loop
    - Human initiates
    - LLM acts
        - with ability to interact with Environment
- Cycle repeats

In this generation, the Harness was analagous to an Operating System.

Just as an OS allows a CPU to
- interact with "peripherals" (disk/filesystem; devices/printers)
- with strict permissions

the first generation Harness allows the LLM (CPU) to 
- interact with the Environment (read/write files; search web)


The Harness provides the environment and services required for the CPU to do meaningful work.

* **System Calls (Tools):** 
    - The LLM doesn't "edit files"; it makes a request to the Harness Kernel.
* **Drivers (MCP):** 
    - The Model Context Protocol allows the OS to talk to external "hardware" like GitHub or Databases.
* **Memory Management:** 
    - The Harness "pages" context in and out of the LLM’s window

In this view:
- The LLM is the **CPU** 
- stateless reasoning

Each time the LLM is called
- it has **no memory** of anything that occurred prior
- **except** for the Context provided by the Harness

For a Chatbot application
- the Harness passed a Context that was an historical record of all past interactions

For an Agent
- the Context that is passed is carefully managed
- to facilitate progress toward a goal.


The new generation of Harness seeks to facilitate *Full Agency*
- Human initiates conversation with Harness
- Harness engages in *multi-turn* conversation with LLM
    - replacing the human while the mission is occuring
- Harness responds to Human with result at conclusion of conversation with LLM

# Defining the "Agent"

A common misconception is that the LLM is the Agent
- but it is Stateless
- so can't assume Agency

It needs
- state (context)
- direction (the mission)

This is what the next-generation Harness needs to provide.

The Agent emerges from a combination:

$$Agent = LLM (Brain) + Harness (OS) + Environment (Reality)$$

## The Functional Roles:
* **The LLM (The Brain):** 
    - Provides the **Reasoning**. 
    - But it is **stateless**
        - depends on the Harness to give it Context
    - It analyzes the current state
    - and predicts the next "System Call."
    
* **The Harness (The OS):**
    - Provides the **Agency**. 
    - It 
        - manages the state
        - enforces the mission
        - executes the tools.
        
* **The Environment (The World):**
    - Provides the **Feedback**. 
        - This is the codebase, the database, or the API the agent is trying to change.

# Harness Engineering




The initial approach to Full Agency
- trained the LLM to mimic Human behavior in solving a task
    - using the same tools as humans
    - Human Computer Interface (HCI)
- relegated the Harness to a supporting role
    - executing tool calls


However, modern research 
- notably from the **Princeton NLP Group** and their work on **SWE-agent**

suggests that
- the **Human-Computer Interface (HCI)** is the wrong metaphor for LLMs. 

Research has shown

> **"Performance is a function of the Interface."**

Using the same model with a better Harness results in better performance on Coding Tasks.

The Agent "works" better when
- the Harness 
- and the Brain 

are perfectly synced in cooperatively achieving a mission.

Achieving this synchronization requires the creation of
-  a specialized **Agent-Computer Interface (ACI)**
- as provided by a  better Harness

*Harness Engineering* is the process of designing a better Harness.
- turning a chaotic, human-centric OS 
- into a structured, agent-centric ACI


Success is measured by 
- how little the LLM Brain has to "think" about the environment
- so it can spend 100% of its reasoning on the **Goal**.

When an agent fails a mission, the questions to ask, that lead to a better Harness are:
 
1. "Did the Harness provide the wrong context?"
2. "Was the tool output too noisy for the Brain to parse?"
3. "Did the Validation check fail to provide a clear enough error message?"

We show how these questions
- have lead to the next-generation Harness
- based on creating an Agent Computer Interface
- that have enabled Agents to succeed in increasingly longer tasks

# Running example

We chart the evolution of the Harness throughout this module with a running example
- Agent used for Coding.

Suppose we encounter an exception

    AuthError
    
during execution.

Our goal is to have our Agent diagnose and correct the underlying issue.



# Human Computer Interface (HCI)  vs Agent Computer Interface (ACI) : Motivation

## The HCI Way (Traditional Shell)

A first step for the Agent in diagnosing the `AuthError` exception is locating it.

An Agent that mimics the process of a human coder
- would use shell commands (e.g., `grep`)

to search for relevant files.


```text
> grep -r "AuthError" ./src
./src/auth/middleware.py:34:    raise AuthError("Invalid Token")
./src/dist/bundle.js:1: var a=function(){if(e)throw new AuthError("b");... [4,000 more characters]
./src/logs/app.log.2024-01-01: [DEBUG] Checking AuthError...
./src/tests/test_auth.py:12: def test_AuthError_handling():
```

What is returned is a "Wall of Text"
- lots of noise

Using visual skills, the human coder quickly filters the Wall of Text to the most-likely cause

    ./src/auth/middleware.py:34:    raise AuthError("Invalid Token")

    

However, an Agent
- must use its own context window 
- to parse file paths, line numbers, 
- and ignore irrelevant minified code or binary noise.

The agent's attention is pulled toward minified JS and old logs.

- **Cost:** 500+ tokens of raw text to find one relevant line.

That which is cheap/natural for a human (visual scanning)
- is expensive/difficult for the Agent
- and consumes a precious resource: context

## The ACI Way (Engineered Harness)

In an ACI, the Harness (**rather than the LLM**)
- intercepts the search
- filters out ignored directories (like dist/)
- ranks the results by "Likelihood of Relevance," 

and returns a Structured Intelligence Object.
- more appropriate for consumption by an LLM than a human


    {
      "status": "success",
      "query": "AuthError",
      "top_signal_match": {
        "file": "src/auth/middleware.py",
        "line": 34,
        "context": "raise AuthError(\"Invalid Token\")",
        "reasoning": "High-signal: Source code definition found in active middleware."
      },
      "other_matches_summary": [
        { "type": "test", "count": 1, "note": "Reference in test_auth.py" },
        { "type": "noise", "count": 15, "note": "Suppressed (logs, minified files, hidden dirs)" }
      ],
      "action_hint": "Use `edit_file` to modify line 34 or `list_dir` to see middleware neighbors."
    }

Result: Pure Signal

The Harness has already "decided" what matters
-  leaving the LLM to focus purely on the fix

- **Cost:** ~60 tokens. (90% reduction in context tax).

In the HCI example, the LLM acts as a Parser.

In the ACI example, the LLM acts as a Decision Maker.

Harness Engineering is about
- moving the "Labor of Parsing" out of the model
- and into the interface.

<table>
    <center><strong>Wall of Text (old way) vs. Curated, Structured Information (new way)</strong></center>
    <tr><img src="images/Harness_ACI_old_vs_new_2.png">
</table>

<table>
    <center><strong>Inside the Harness ACI Layer<br>Discovery (describe_source to query schema)<br>Summarizer<br>Ranking<br>Permissions<br>followed by extraction/filtering
        </strong></center>
    <tr><img src="images/Harness_ACI_old_vs_new.png">
</table>

## The Fallacy of the "Human" Metaphor

Standard operating environments
- Bash shells, Python REPLs, File Explorers

were designed for the **Human-Computer Interface (HCI)**

They prioritize 
- visual feedback
- synchronous streaming
- manual navigation.

When we force an Agent into an HCI, we introduce "Interface Friction":

* **The Noise Tax:** 
    - Humans can ignore ANSI colors or scrolling metadata
    - LLMs pay for that noise in tokens and attention loss
    
* **The Navigation Tax:** 
    - `cd` and `ls` are spatial metaphors that consume reasoning cycles.
    
* **The Formatting Tax:** 
    - Plaintext logs are often "unstructured,"
    - forcing the LLM to waste "brain power" parsing strings instead of making decisions.

# Harness: the next generation

The Harness evolves from an OS into an **Executive** or **Orchestrator**

- ensures that the Mission is completed
- by directing the Brain

It acts a Project Manager
- with the Brain as worker

The Harness
- asks the Brain to design a plan  to solve the task (the Mission)

The Brain
- is *no longer responsible* for the plan
- it takes direction from the Harness

The Harness
- maintains a global state
    - steps of the plan that have been completed/remain to be done
- parcels out steps for the Brain to work on

The  **The Harness (Project Manager):** 
- enables the Brain to succeed
- by limiting its view
- enabling it to focus on Reasoning


It does this by
 **Step-Gating**:
1. **Isolation:** 
    - The Brain only sees the context for the *current* step.
2. **Validation:**
    - The Harness won't let the Brain move on to the next step
    - until *Validation checks* succeed
        - the **Linter** and **Sandbox** confirm the current step has succeeded
3. **Recovery:** 
    - If the current step fails to validate, the Harness can
        - ask the Brain for a "Sub-Plan" to fix the local error
        - without discarding the entire Global Mission.

> The Brain provides the logic, but the Harness provides the **Continuity.**



##  Defining the "Mission": The Executive Anchor

In Harness Engineering, the **Mission** 
- is the stateful objective
- that governs the agent's life cycle.

It is the "North Star" that the Harness  uses to evaluate every action the agent takes.

A **Mission** is the high-level invariant that the Harness protects.


### Anatomy of a Mission

1. **The Primary Objective:** 
    - A clear, non-negotiable statement of success 
        - e.g., "Implement the 'Forgot Password' flow"
2. **The Guardrails:** 
    - Operational constraints enforced by the Harness 
        - e.g., "Do not spend more than $2 in tokens"
        - e.g., "Do not modify the `main` branch"
3. **The State:**
    - A live, persistent record of progress
    - Persists across hundreds of tool calls.

Here is the **Mission Statement** for the Harness (using our `AuthError` exception example)

**The Mission Statement**

> **Objective:** Identify, root-cause, and patch the `AuthError` exception currently affecting the production login flow.
> 
> **Constraints:**
> * Operate exclusively within the `/staging` sandbox.
> * Achieve a 'Pass' from the **Linter** before requesting deployment.
> * **Goal:** Resolve the system state; do not "discuss" the bug.

### The "Drift" Problem & The Harness Solution

Without a Mission-aware Harness
- agents suffer from **Cognitive Drift**. 

They get distracted by sub-tasks
- like fixing a typo in a log
— and lose sight of the original bug they were sent to find. 



A Goal is what the LLM *thinks* it is doing. 

A Mission is what the Harness *ensures* gets done.

The Harness is the difference between 
- an AI that 'tries'
- and an Agent that 'delivers'.

### The Lifecycle of a Mission: From Exception to Resolution

We follow the Agent through  the
 > **Fixing the `AuthTokenExpiredException`.** 
 
In this workflow
-  the "Conversation" is handled entirely by the **Harness (OS)** 
- providing feedback to the **LLM (CPU)**
- based on the Mission Statement.



| Phase | Agent Intent (The Brain) | Harness Role (The OS) | The ACI Mechanism |
| :--- | :--- | :--- | :--- |
| **01. Discovery** | "Where is the error coming from?" | **Filters the Haystack:** The Harness executes a search but caps results to the top 10 matches to prevent context flooding. | `file_search()` |
| **02. Navigation** | "Show me the logic around the token refresh." | **Manages Vision:** The Harness opens the file at the relevant line and maintains the "Scroll State" for the Agent. | `stateful_view()` |
| **03. Intervention** | "I'll update the expiration buffer from 5m to 30m." | **Immune Response:** The Harness intercepts the write and runs a Linter to ensure the "Mission State" isn't corrupted by a syntax error. | `linter_gate()` |
| **04. Verification** | "Is the system stable now?" | **Isolated Validation:** The Harness executes the test suite in a restricted environment and reports the success back to the Agent. | `sandbox_exec()` |

# Key elements of the new generation

There are several innovations
- Agent Computer Interface (ACI)
- Context Management Philosophy
- Other Key Executive functions

## The Interface (ACI)


The tools used by the Mission Aware Harness
- are different than those used by a Human

The Princeton group showed that
- the Human Computer Interface (HCI)

*impedes* performance.

What is needed is a new *Agent Computer Interface*
- optimized around the needs of an LLM

### File Search (High-Resolution Discovery)
Unlike a standard `grep`, the **Agent-Centric Search** is designed for a token-constrained brain.

* **The Filter:** 
    - It ignores non-code assets (images, binaries) by default.
* **The Cap:** 
    - It returns "Snippets" rather than full files
        - ensuring the LLM isn't flooded with 5,000 lines of search results in a single turn.

### Stateful File Viewer (Focused Vision)

An LLM cannot "look" at a 10,000-line file without drowning in noise. 

The **Stateful Viewer** acts as a "Slit-Scan."
* **The Window:** 
    - Shows only 100 lines at a time.
* **The State:** 
    - The Harness remembers the agent's "Scroll Position."
* **The Benefit:**
    - Constant token costs and zero "Attention Drift."

### 4.2 The Linter (The Immune System)

When the Agent proposes a fix for the `AuthTokenExpiredException`, 
- the Harness intercepts it.

The Harness applies "quality control" to the LLM's proposed solution

* **The Gate:**
    - If the code has a syntax error, the Harness rejects the write.
* **The Feedback:** 
    - The Agent and Harness "argue" 
    - until the code is syntactically perfect
    - before it ever touches the disk.

###  Security & Permissions (The Sandbox)

The Harness enforces a **Least Privilege** model. 

Even if the Agent attempts to delete the root directory,
- the OS blocks the call

ensuring safety.



## Philosophy: Selective Context

The components of the ACI that we introduced reflect a particular "philosophy" of Context Management.

In traditional Human-Computer Interaction (HCI)
- we value **transparency and history accuracy**
- as a consequence: the Context at each turn of the LLM is *historically accurate*

Moving to ACI
-  we value **focus and state**. 
- as a consequence, Context at each turn
    - is the exact set of information needed to complete the turn
    - **not** a history of past turns

The governing philosophy is:
- **The Brain should only see what is necessary to take the next correct step.**

Thus the Context provided to the LLM
- is no longer a **Diary**
    - un-edited transcript of the entire preceding conversation
- it is a Curated **Briefing document**
    - supplying only immediately relevant facts and background
    
| Concept | Chat Agent (HCI) | Harness-Managed (ACI) |
| :--- | :--- | :--- |
| **Memory** | Chronological Diary | **Synthetic Briefing** |
| **History** | Sacred and growing | **Disposable and pruned** |
| **Reality** | Raw and unfiltered | **Curated by the Harness** |


## Other Key Executive functions

These fall into the broad description of "Keeping on Mission"
- Making sure that the Mission Statement is not lost in a long conversation
- Preventing endless looping
    - where LLM is unable to solve problem
- Keeping focus by distilling large number of small details into "Golden Nuggets" of information



# Agent Computer Interface

We create *Agent-centric* tools
- that replace *Human-centric* equivalents

A primary difference
- Humans have sensory skills (visual scanning)
- that the LLM can only mimic at high cost
    - consume large amounts of context



##  From `grep` to `file_search`

We can illustrate the difference between Quantity vs Quality using our original example
- finding files in our code base that may have raised the runtime `AuthError` exception

The Human-Centric approach: use `grep` to identify lines in relevant files

**Standard Shell (Raw Data):**

```bash
grep -r "AuthError" ./src
# Output: 500 lines of raw text, 4000 tokens used.
# Result: LLM gets "lost in the middle" of the search results.

```


- Returns a stream of text
- Result is like a "Token Firehose"
    - * One search for "handle_click" might return 2,000 lines of minified JS, logs, and build artifacts.

- Low Signal to Noise

The Agent-Centric approach: an *intelligent* search tool
- `file_search` tool
- doesn't just search
- it **ranks and summarizes**

returning *relevant* information.

And, importantly, returns a *structured result* rather than a stream of tokens.

    # The Harness "pre-chews" the data
    def file_search(query):
        raw = system_grep(query)
        # 1. Filter out non-source code (logs, .json, .md)
        # 2. Rank by "Likely Importance" (Controllers > Configs > Tests)
        # 3. Return a "Map" rather than "The World"
        return {
            "top_match": {"file": "./src/auth/middleware.py:", "line": 34},
            "others": ["./src/dist/bundle.js", "./src/logs/app.log.2024-01-01"],
            "message": "Found 45 total matches. Summarizing for clarity."
        }


## The Stateful File Viewer: "Managed Vision"

A **Stateful File Viewer** 
- is a specialized ACI tool 
- that solves the **Context Flooding** problem. 

It allows an agent to navigate massive files without "drowning" in tokens.

Unlike the `cat` command (which dumps everything), a Stateful Viewer acts like a **Window**:
1. **Open:** 
    - The Agent opens a 10,000-line file.
2. **View:**
    - The Harness shows only lines 1-100 + a "Status Line" (e.g., *Total Lines: 10,000*).
3. **Navigate:** 
    - The Agent issues commands like `scroll_down` or `search("API_KEY")`. 
4. **Update:**
    - The Harness shifts the window and only provides the new 100 lines of context.

**What is the "State" in Stateful Viewing?**

In a standard Human interface (HCI), **State** is 
- maintained by the Software (the scrollbar)
- and the Human (the brain).

In an ACI, the **Harness** must own the state.

**The State consists of:**
1. **The Cursor:** 
    - Where is the Agent currently looking? (Line numbers).
2. **The Context:**
    - Which file is currently active?
3. **The Buffer:** 
    - What was the last thing the Agent saw?
    


**Anatomy of a Stateful Mission Loop**

Here are a sequence of states encountered while the Agent navigates file `server.py`


| Step | Agent Command | Harness Action (The State) | Output to Agent (The Context) |
| :--- | :--- | :--- | :--- |
| **01** | `open_file("middleware.py")` | Sets **Pointer** to Line 1; **File** to `server.py`. | "Viewing lines 1-100 of 4,000." |
| **02** | `scroll_down()` | Moves **Pointer** from 1 to 101. | "Viewing lines 101-200. (2% of file)." |
| **03** | `search("class Auth")` | Scans file; updates **Pointer** to 2,450. | "Found at line 2,450. Viewing 2,450-2,550." |
| **04** | `read_file()` | *Remembers* the current window position. | Re-displays 2,450-2,550 for reference. |

**Crucially**:

State
- is maintained by the **Harness**
- but injected into the LLM context on every turn

When the LLM wakes up for the next turn
- the first thing it sees is a structured block generated by the Harness:

        STATUS: You are currently viewing middleware.py.
        WINDOW: Lines 142-242 of 600 total.
        LINTER: PASS.

**State** is what allows the Brain (LLM)
- to **remember** where it is
- between turns

Making sure that the LLM 
- knows, unambiguously, where it is in the file
- prevents **Context Drift**.

By having the Harness "remind" the Agent of its position every turn
- we eliminate the need for the Agent to try and remember its own coordinates
- enabling the Agent to **maintain focus on its Mission**


Before we had stateful viewers
- an Agent had to "re-read" the entire file
- every time it wanted to see a different part. 

Apart from being expensive
- The Agent would lose track of which "version" of the file it was looking at

Statefulness allows the Agent to
- "navigate" rather than just "consume."

It transforms the file system from a pile of documents into a 3D space the Agent can move through.

The State turns the LLM from a "reader" into an "explorer".




| Feature | Stateless (`cat` / `read`) | Stateful (`file_view`) |
| :--- | :--- | :--- |
| **Token Usage** | Exponential (dumps entire file). | Constant (only the "window"). |
| **Focus** | Low (distracted by noise). | High (focused on relevant lines). |
| **Navigation** | Disorienting (must re-read). | Natural (remembers "where" it is). |



## The Linter: The Agent's "Immune System"

A **Linter** is 
- a static analysis tool 
- that checks code for errors 
- without executing it. 

**How the Linter-Gate Works**

1. **The Proposal:** 
    - The Agent attempts to save a file:
    
        `write_file('auth.py', content=...)`.
2. **The Intervention:**
    - Before the file hits the disk
    - the Harness intercepts it
    - and runs a **Linter**.
3. **The Verdict:** 
    - **Pass:** The file is saved.
    - **Fail:** The Harness **rejects** the change and feeds the error logs back to the Agent.

The Linter acts as a **Mandatory Gatekeeper**. 
- similar to a spell-checker in concept

It makes sure the Mission is not derailed by syntax issues.

It operates as an "Immune System" for the Agent.


Originally, the linters used
- were standard tools
- "Detect syntax errors"

but they evolved (via Self Improvement -- see Level 2 Recursion below).
- into custom checks
    - whose rules are defined by the Agent
- that were *created by the Agent* to make itself more effective

Consider a "logic checker" that rejects
- API calls that are technically legal
- But prone to failure/bugs
    - frequent, non-compatible version changes of the library


## Security and Permissions

The Harness (via tools) is now able to change 
- the environment
    - local machine state
    - external state
        - perform a financial transaction

It is the responsibility of the Harness to provide **Safety Rails**
* enforce **State-Changing Permissions**.

**Tiered Governance:**
1. **Read-Only:** No approval needed (Search, List).
2. **Write-Proposed:** Human-in-the-loop (Harness shows a Diff, waits for `Y/N`).
3. **Action-Blocked:** Restricted commands (e.g., `rm -rf`, `curl` to unknown IPs).


**Sandboxing:** 
* The harness creates a "Simulated Reality" for the agent
* preventing it from ever "seeing" or "touching" 

the host system's sensitive core.



##  Virtualizing the Context Window

Just as a Hypervisor 
- allows a Virtual Machine to access more RAM than it physically has

the Harness uses 
- **Context Orchestration** 

to give the LLM the illusion of a massive, perfectly-organized memory.

The Harness performs **"Lossy Compression with Intent"**:
1. **Intercepts** the environment's raw, 50MB response.
2. **Filters** for the "Golden Nuggets" relevant to the current task.
3. **Presents** a structured, high-SNR summary to the Brain.

# Philosophy: The "Selective Context" Doctrine





The key insight behind Harness Engineering is refuting the assumption

>"Infinite Context windows (1M+ tokens) solve all problems."

The reality is that there is an **"Attention Tax."**
* As context length $n$ increases
* the model's ability to retrieve the "needle" in the "haystack" decreases.
* Quadratic complexity of Attention: $O(n^2)$.

In the old view
- finite context is the issue
- which is solved by Compaction
    - ** Shrinking data to fit a limit (Brute force)
        - replacing chunks by summaries
        
The new view
- **Relevance**is important
- Solution is *curation*
    -  Ensuring the 1,000 tokens the model sees are the *right* 1,000 tokens.

We can summarize the new view:

**The Harness Goal:** Maximize the Signal-to-Noise Ratio ($SNR$).

Quality over Quantity

## Context: From "Diary" to "Briefing"

The hypothesis for the inability of early Agents to succeed (particularly at long-running tasks)
- was that they were treated like Chatbots. 
- They were given a Context that was a "Diary"
    - a chronological, unedited history of every mistake, typo, and 404 error they encountered.

We have evolved into Context as a **Briefing**
- The Harness performs **Context Re-Writing**.
- It "wipes" the messy history and replaces it with a synthesized "Mission State."

### The "Wipe and Replace" Mechanism

The Harness acts as an "Executive Editor" for the Brain's memory
- Every few turns, it performs a **State Consolidation**:
    1. **Pruning:** 
        - It deletes raw tool outputs and terminal logs that are no longer needed.
    2. **Distillation:** 
        - It collapses 10 turns of trial-and-error into a single "Lesson Learned."
    3. **The Clean Slate:** 
        - It presents the LLM with a fresh "Dashboard" that makes the Agent feel like it has been perfectly efficient from the start.

| Philosophical Pillar | Goal | ACI Technique |
| :--- | :--- | :--- |
| **Amnesia as a Feature** | Prevent "Context Drowning" | Periodic history wiping / Log pruning. |
| **Synthesized Truth** | Consolidate progress | Rewriting the "Mission Log" summary. |
| **Environment Primacy** | Prioritize "Now" over "Then" | Injecting **State** headers (Cursor, File, Linter). |

### Selective Context: A Curated Reality

To maintain the "Briefing," the Harness manages three specific filters:
* **The Filter of Relevance:** Hiding files/data unrelated to the current sub-task.
* **The Filter of Success:** Only carrying forward successful logic, not the "garbage" of failed attempts.
* **The Filter of Scale:** Using **Stateful Windows** so a million-line project feels like a 100-line task.

**Selective Context** is the act of
- engineering a perfect workspace for a stateless mind


## Sub-Agents to boost Relevance:  "Context Firewalls"

Using multiple sub-agents to solve a task
- seems, on the surface, to be about parallelism/speed

but it is also a key tool in making Context relevant.

* **The "Mental Clutter" Problem:** 
    - If an agent spends 20 turns trying to find a bug
    - its context is filled with "failed attempts" and "error logs."
    - *Confusing* as well as consuming finite context
* **The Solution: Ephemeral Sub-Agents:**
    - * Spawned for a single, narrow task.
* **Isolation:** 
    - The sub-agent has its own context window.
* **Garbage Collection:** 
    - Once the task is done, the sub-agent is killed. 

**Key**:

Only the **Success Result** is passed back to the Main Agent.
- the intermediate context of the sub-agent
- doesn't *distract* (or consume context) the main Agent

<table>
    <center><strong>Delegation to Subagents</strong></center>
    <tr><img src="images/Harness_Subagents.png">
</table>

**Math of Context Efficiency:**

If a task generates $T_{noise}$ tokens, 
- using a sub-agent ensures:

$$\text{Context}_\text{Main} = \text{Context}_\text{Initial} + \text{Result}_\text{Summary}$$



The $\text{Context}_\text{sub-agent}$ is never "paid for" by the Main Agent's attention mechanism.


## Standardizing the Interface: MCP

Model Context Protocol (MCP) defines
- a universal interface
- for connecting any Tool

to a Harness.

By standardizing how the Harness connects the Brain (LLM) to the World (Tools) it acts like

* The "USB-C" of AI.

It operates via **Dynamic Context Injection** (like all tool calls)
- A tool call 
    - to an MCP server
    - rather than a local tool
- returns a result
- which is injected into the Brain (LLM context)

The Agent (Host)
- doesn't need to know how to solve a sub-task/have access to restricted data

The MCP call is kind of an *LLM subroutine* providing encapsulation.

Again
- the context of the MCP server is ephemeral
- doesn't confuse or consume context in the Host

<table>
    <center><strong>Model Context Protocol (MCP) Stages <br>Discovery (of relevant servers)<br>Return curated resluts</strong></center>
    <tr><img src="images/Harness_MCP
.png">
</table>

### Managing Context: MCP vs. Sub-Agents

Both MCP and Sub-agents off-load the work.

But there is a potential difference in *Context Hygiene*

MCP
- Agent (Host) can't control what the MCP server returns
    - length: may flood the Agent context
    - format: may not be structured

Sub-agents
- Agent can control
    - length
    - format

by giving the sub-agent a "Mission"
- which includes instructions on length ("summarize") and format (structured output)


| Executive Function | Naive View (Script) | Engineered View (Executive) |
| :--- | :--- | :--- |
| **Logic Flow** | Blindly follows next token. | Recognizes patterns and halts loops. |
| **Context Management** | Cumulative (Drowning in noise). | Curated (High Signal-to-Noise). |
| **Error Handling** | Reports raw error to LLM. | Pre-analyzes error and suggests fixes. |

The LLM provides the **Intelligence**
- but the Harness provides the **Continuity**

It ensures that the 'Agent'
- remains a consistent personality
- working toward a single goal

rather than a series of disconnected, stochastic reactions.

# Other Key Executive Functions

* **Progress Tracking (The North Star):**
    - Since LLMs can "forget" the original goal during long tasks 
    - the Harness continually re-injects 
        - the **Mission Statement**
        - and current progress
        
    into the system prompt.
    
* **Loop Detection (Circuit Breakers):** 
    - The Harness monitors tool-call history. 
    - If it sees the same `ls` command three times with the same result
        - it intercepts the loop and forces the model to pivot.
        
* **Context Hygiene (The Janitor):**
    - Proactively prunes the context window. 
    - Rmoves the "scaffolding" of failed attempts and search noise
        - keeping the model's focus on the **Golden Nuggets** of relevant data.


# Case Study: Claude Code as a Harness

**Claude Code** is a prime example of an engineered **Agentic Harness**. 
- It transforms a standard CLI
- into a specialized **ACI (Agent-Computer Interface)**

optimized for the Claude 3.5 Sonnet model
- in theory: can plug in any model
- that has been trained in tool use
- and the protocols (Tool use/MCP/Skills)

implemented by the Harness

## How Claude Code functions as a "Hypervisor":

1. **The Toolset (The Hands):** 

    * It provides the model Agent-Centric (ACI) tools: `file_search`
    * rather than Human-Centric tools: `grep`, `ls`, and `file_edit`
    
2. **Context Orchestration (The Filter):** 
    * It manages the model's context window by 
    * automatically summarizing long outputs 
    * and using "paging" for large files.
    
3. **State Management (The Memory):**
    * It tracks the "state" of the repository
    *  allowing the model to remember which files it has already edited
    * without needing to re-read them every time.

# Recursive Self-Improvement vs. Iterative Bootstrapping


*Recursive Self-Improvement (RSI)* is a holy grail in AI
— the "intelligence explosion" where 
- an AI writes a better version of itself
- which then writes an even better version, ad infinitum.

A common question is whether Claude Code and the 3.5 family represent **Recursive Self-Improvement (RSI)**.

The answer is nuanced: we are currently in an era of **Model-Aided Evolution**.

## The Recursive Harness

Claude Code represents a recursive loop in **Productivity**. 
* Engineers use Claude to write the ACI (Agent-Computer Interface).
* A better ACI allows Claude to be more productive.
* That productivity is used to accelerate the development of the next model.

##  The "Intelligence Explosion" vs. "Efficiency Gains"

We have not yet reached a closed-loop "Intelligence Explosion" 
- where the AI independently discovers new architectures. 

Instead, we are seeing **Efficiency Compounding**:
* **RLAIF:** AI feedback loop for alignment.
* **Synthetic Context:** AI-generated "Golden Nuggets" for fine-tuning.
* **ACI Refinement:** AI-driven optimization of its own "limbs."

We use the term "Levels" to refer to
- the part of the system 
- that is being modified by AI

| Level | Type | Current Status | Example |
| :--- | :--- | :--- | :--- |
| **Level 1** | **Recursive Data** | **Standard** | Using Claude 3 to generate high-quality synthetic training data for Claude 3.5. |
| **Level 2** | **Recursive Tools** | **Active (Claude Code)** | Using the Agent (Claude) to write the code for its own ACI/Harness and tool-wrappers. |
| **Level 3** | **Recursive Architecture** | **Theoretical** | The AI autonomously rewriting its own transformer math, loss functions, or silicon chip architecture. |

### Level 2: The "Tool-Builder" Phase

**Status (mid-2026)**

We are currently transitioning from **Level 1** (Data Loops) into **Level 2** (Tool Loops). 

Level 2 is the stage
- where the AI stops being just a "worker" 
- and starts being the **architect of its own laboratory.**

While Level 3 represents the "Hard Takeoff" scenario
- Level 2 is where the most significant Engineering gains are happening right now.


**Why Level 2 is a Step-Function in Capability:**
1. **The Insight:** 
    - The Agent realizes a task is failing because the interface is noisy (e.g., raw Bash output).
2. **The Creation:** 
    - The Agent writes a specialized tool (an MCP server) to filter and structure that output.
3. **The Deployment:** 
    - The Agent "installs" its new tool into the Harness.
4. **The Result:** 
    - The Agent is now 10x more effective at that task, despite its "IQ" (the weights) remaining the same.

    

# Quantitative  results: why ACI is better than HCI

The Princeton paper proved demonstrated
- the same Brain (LLM)
- performs *much better* in an ACI Harness than a HCI Harness


## The Original 2024 Result (SWE Bench evaluation)

When the same model (GPT-4) was tested on real-world GitHub bugs, the "Interface Premium" was undeniable:
* **Naive Shell (HCI):** 1.6% Success Rate.
* **SWE-agent (ACI):** 12.5% Success Rate.
* **The Multiplier:** A **7.8x improvement** just by redesigning the environment.

**Conclusion**

- We don't need a 10x bigger model to get 10x better results
- We need a 10x better Harness

## The 2026 Landscape (SWE-bench Verified evaluation)

Today, we see that the combination of 
- **Recursive Tools** 
    - Level 1 (non-recursive): human engineered transition from HCI to ACI
    - Level 2 (recursive): generation $g$ of the ACI Agent creates generation $(g+1)$ ACI Agent
- **High-Reasoning Models**

has pushed success rates above 75%.

| Agent Architecture | Success Rate | Efficiency Note |
| :--- | :--- | :--- |
| **Original ACI (2024)** | 12.5% | First proof of ACI theory. |
| **Modern Harness (2026)** | 76.8% | Leverages "Thinking" models + Recursive ACI. |
| **Gemini 3 Flash (Optimized)**| 75.8% | Provenance of the "Cheap & Fast" Agentic Era. |

**Message from my AI Assistant**

>Gemini 3 Flash (the model you are interacting with now) is nearly equal in performance to the most expensive "Opus" models while being significantly cheaper. This is a direct result of Harness optimization—the interface has become so efficient that smaller, faster models can now outperform the giants of the previous year.


## The December 2025 "Step-Function"

In late 2025
- the AI industry observed a dramatic shift in the **Task-Completion Time Horizon**.
    - length of a task that AI could solve
    - measured in time/number of steps
- This was not the result of a single "Scale" increase
- but the widespread adoption of **Level 2 Recursive Harnessing**.

This has been called the "Holiday Breakthrough" in developer circles
- the moment when AI agents crossed the chasm 
- from "unreliable interns" 
- to "autonomous project leads."

### The "48-Hour Agent" Breakthrough

* **Old Limit:** 
    - 5–15 steps (~10 minutes)
    - Past this, agents would "hallucinate" themselves into a loop or lose track of the Mission.
    - This is called **Recursive Degradation**
        - the longer the task, the more likelyan Agent was to loop or hallucinate.
    
* **New Reality:** 
    - 500+ steps (2–3 days). 
        -Agents can now manage full-cycle migrations:
        - from auditing code
        - to writing PRs, running CI/CD, and fixing deployment regressions.
        
| Metric | Pre-Dec 2025 | Post-Dec 2025 |
| :--- | :--- | :--- |
| **Stability Window** | 5-10 Minutes | **Hours / Days** |
| **Interface** | Raw Terminal (HCI) | Agent-Computer Interface (ACI) |
| **Logic** | Scripted Function Calls | Recursive Tool Use (Level 2) |


### The Mechanics of the Leap

1. **Circuit-Breaking Harnesses:**
    - The Harness now detects "Infinite Loops" 
        - e.g., the agent trying the same `ls` command
    - and intervenes before the context is poisoned.
2. **Lossy Context Summarization:** 
    - Instead of carrying 100% of the logs
    - the Harness uses "Selective Forgetting" 
        - to keep only the **Golden Nuggets** relevant to the Mission.
3. **Agentic Tool Discovery:** 
    - The agent can now 
        - realize it lacks a tool
        - write an MCP server to create that tool
        - and then use it to finish the task.
    - This is Level 2 Recursion in action

A key contributor to this step-function improvement was the migration from
- Level 1 Harness (Human-Engineered)
    * **Example:** Standard MCP servers, basic RAG, hard-coded "Circuit Breakers."
    * **Role:** Humans build a better "car" for the AI to drive.
    * **Result:** Increased reliability, but limited by the human's ability to predict every failure.

- to Level 2 Harness (AI-Recursive)
    * **Example:** Agentic Tool Discovery (AI writing its own tools), Autonomous Context Pruning.
    * **Role:** The AI "pulls over," opens the hood, and **tunes the engine** while mid-Mission.
    * **Result:** A "Step-Function" in capability. 
        - The Agent can now adapt to environments the human engineer never envisioned.




At the start of 2025
- we were trying to keep the AI on track with better prompts

By Winter break
- we started building **Operating Systems for Intelligence**.

The 'Step-Function' wasn't the AI getting smarter
— it was the environment getting **safer for the AI to work longer**."

# The AI Laboratory: From Tool to Scientist

Andrej Karpathy and others describe the current frontier of AI as
- **"Building the Laboratory."** 

This represents the shift from 
- a model that simply "answers questions"
- to an agent that "manages its own improvement."

It describes an 
**Automated Research Cycle**. 

The Harness is 
- the Laboratory
- the LLM is the Lead Scientist
    - perhaps managing a team of Junior Scientists (sub-agents)
    - who share results
        - git commit
- Every time the Agent fixes a bug in its own Harness
    - the Laboratory becomes more advanced."

##  The Components of the Lab

* **Synthetic Data Generation:** 
    - The model creates its own "textbooks" to learn from.
* **Tool-Building (The ACI):** 
    - The model identifies where its "hands" are clumsy 
    - and helps engineers (or itself) write better Harness code.
* **Self-Correction Loops:** 
    - The model runs a task, observes the failure in the environment
    - and updates its "Skill" (muscle memory)
    - to avoid the error next time.

##  Evolution of the Concept

While popularized by Karpathy’s "LLM OS" framework, this is a culmination of several historical threads:
1.  **Self-Play (DeepMind):** The core logic of AlphaGo applied to language.
2.  **RLAIF (Anthropic/OpenAI):** Using AI to grade and improve AI.
3.  **Harness Engineering:** Creating a structured environment (The Lab) where the model can safely interact with the "real world."



##  Is ACI Only for Code? (The "Brittle" vs. "Lossy" Debate)

It is easy to assume that Harnesses are
- only for coding because code is **brittle**
    - unforgiving syntax
    
Natural language is **lossy** (less precise) but forgiving. 

However, ACI
- **is not about syntax**
- **it is about Grounding.**

and thus applies to non-coding tasks as well.

### The Translation of Concepts:

Consider an Agent for Legal tasks.

* **The Coding Linter** → **The Logic/Fact Checker:**
    - acts as a fact-checker
        - validates citations against hallucination
   
* **The Codebase** → **The Knowledge Base:** 
    - Whether the "Laboratory" is 
        - a folder of `.py` files 
        - or a folder of `.pdf` legal briefs
        
    - the ACI provides the structured way the Agent interacts with that world.
* **Level 2 Recursion:** 
    - An agent realizing it needs a better "Search Tool" for legal precedent
        - to find specific clauses in a contract    
    - builds a custom search-filter for itself.

### The Universal Necessity of ACI:

The "Step-Function" increase in reliability we saw in December 2025 
- applies to **any long-horizon task**. 

1. **The Brain (LLM)** provides the creative strategy.
2. **The Harness (ACI)** provides the **Execution Guardrails.**



# Summary & Future Outlook

* **The Princeton Legacy:** 
    * SWE-agent proved that the **Interface (ACI)** is the primary driver of performance.
* **The Industry Shift:** 
    * Tools like **Claude Code** are the first mainstream "Production Harnesses."
* **The "Harness Engineer" Role:** 
    * It's not just the model that determines performance
        * the Harness is critical

Harness Engineering is about
* building the **Context Filters**
* and **Safety Exoskeletons** 

that make the Brain (LLM) perform at a higher level.

---

In [2]:
print("Done")

Done
